<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. Centralized Governance with UC with External Compute Engines</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 4.1 Lecture: Centralized Governance with UC with External Compute Engines

This lesson covers the second major interoperability use case, where transformation and write workloads run on external compute engines like EMR or Flink while Unity Catalog continues to enforce governance over the data they read and write.

## Learning Objectives

By the end of this lesson, you will be able to:
- Describe the centralized governance pattern with external compute
- Identify usage scenarios for external compute with UC governance
- Explain the implementation steps for EMR and similar engines
- Compare this pattern with the centralized processing pattern

## A. Context

In this pattern Databricks is no longer the compute hub: external engines do the heavy lifting and UC remains the single point of governance over the underlying tables.

<div class="mermaid" id="diagram-4-1-external-governance" style="font-size: 1em;">
flowchart TB
    UC["<b>Unity Catalog</b><br/>Central Governance"]
    UC -->|"Credential<br/>Vending"| EMR["<b>AWS EMR</b><br/>ETL Processing"]
    UC -->|"Iceberg REST"| Flink["<b>Apache Flink</b><br/>Streaming"]
    UC -->|"Iceberg REST"| Spark["<b>External Spark</b><br/>Batch Processing"]
    EMR --> Storage["<b>Cloud Storage</b><br/>UC Managed Tables"]
    Flink --> Storage
    Spark --> Storage
    style UC fill:#FF3621,stroke:#CC2B1A,stroke-width:2px,color:#fff
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });
await new Promise(r => requestAnimationFrame(r));
try {
  await mermaid.run({ querySelector: "#diagram-4-1-external-governance" });
} catch(e) {
  await new Promise(r => setTimeout(r, 1000));
  await mermaid.run({ querySelector: "#diagram-4-1-external-governance" });
}
document.querySelectorAll('#diagram-4-1-external-governance svg text, #diagram-4-1-external-governance svg .nodeLabel, #diagram-4-1-external-governance svg foreignObject div, #diagram-4-1-external-governance svg span').forEach(el => { el.style.fontSize = '1em'; });
</script>

**Key Difference from Use Case 1:** Unlike centralized processing, here **transformations and updates are performed by external compute** (not on the Databricks platform). UC serves purely as the governance and metadata layer.

## B. Usage Scenarios

| Scenario | Description |
|----------|-------------|
| **Data Scientists using EMR** | Specialized processing on EMR while maintaining UC governance |
| **Compliance Teams** | Enforcing consistent data security regardless of compute location |
| **ETL on External Engines** | Running ETL processes that update UC-managed tables |
| **BI via Snowflake** | BI teams querying through Snowflake while adhering to centralized access policies |

### Goals
- Establish Databricks Unity Catalog as the **central governance and metadata** management layer
- Enable external compute engines to **perform transformations directly** on UC managed Iceberg tables
- Maintain **unified governance** while leveraging specialized compute capabilities

## C. Reference Implementation (Writing UC Data from EMR)

<p style="font-size: 1em; line-height: 1.6; color: #333">Setting up centralized governance with external compute mirrors the centralized processing pattern but adds write privileges, since the external engine will be modifying UC-managed tables.</p>

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #37474f; font-size: 1.1em">Prerequisites</strong>
            <p style="margin: 8px 0 0 0; color: #333">A user or service principal with the Account Admin role enabled, and an equivalently privileged role in the external system.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 16px 0">

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 1</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Enable External Data Access</div>
    <div style="color: #455a64">Turn on external Iceberg REST access at the metastore level.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 2</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create Managed Tables</div>
    <div style="color: #455a64">Materialize as managed Iceberg, or as Delta with UniForm.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 3</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create a Service Principal</div>
    <div style="color: #455a64">Provision the SP and a client secret that the external engine authenticates as.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 4</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Grant Privileges</div>
    <div style="color: #455a64">Grant <code>EXTERNAL USE SCHEMA</code>, <code>SELECT</code>, <code>MODIFY</code>, and <code>USE</code> on the catalog and schema to the SP.</div>
  </div>

  <div style="background: #F2A33C33; border: 2px solid #F2A33C; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #A35E0F; text-transform: uppercase; margin-bottom: 4px">EMR &middot; Step 1</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Provision EMR Cluster</div>
    <div style="color: #455a64">Stand up an EMR cluster with Spark and include the Iceberg runtime libraries.</div>
  </div>

  <div style="background: #F2A33C33; border: 2px solid #F2A33C; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #A35E0F; text-transform: uppercase; margin-bottom: 4px">EMR &middot; Step 2</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Configure Spark for UC</div>
    <div style="color: #455a64">Set the REST catalog URI, token URI, and SP client ID/secret in Spark config.</div>
  </div>

  <div style="background: #F2A33C33; border: 2px solid #F2A33C; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #A35E0F; text-transform: uppercase; margin-bottom: 4px">EMR &middot; Step 3</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Run ETL Against UC Tables</div>
    <div style="color: #455a64">Reference UC tables as targets in Spark ETL jobs and write back through the REST catalog.</div>
  </div>

  <div></div>

</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #e65100; font-size: 1.1em">MODIFY Privilege</strong>
            <p style="margin: 8px 0 0 0; color: #333">Unlike the read-only pattern, this scenario requires the <code>MODIFY</code> privilege on the schema because the external engine is performing <b>write</b> operations.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #00695c; font-size: 1.1em">Other Systems</strong>
            <p style="margin: 8px 0 0 0; color: #333">The same pattern applies to other Iceberg-compatible compute engines such as standalone Spark on Kubernetes, Dataproc, Trino, and Flink. Each system has its own configuration syntax for the Iceberg REST catalog and credential setup.</p>
        </div>
    </div>
</div>

## Key Takeaways

- UC provides **centralized governance** even when compute is external
- External engines need `MODIFY` privilege for write operations (unlike read-only access)
- The **Iceberg REST Catalog** + credential vending enables secure external compute
- All operations are **audited and governed** by Unity Catalog regardless of compute engine

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>